# 📦 Google Drive ZIP Extractor for Colab

**Extract ZIP files directly inside Google Drive — no download, no re-upload.**

This notebook mounts your Google Drive, finds every `.zip` file in the folders you specify,
and extracts them back into Drive. Files of any size (tens or hundreds of GB) are streamed
member-by-member so local disk and RAM stay small.

## Quick Start
1. Set your options in the **Config** cell below (at minimum, set `SOURCE_PATHS`).
2. **Runtime → Change runtime type → CPU** (no GPU needed).
3. **Runtime → Run all**.
4. Authorise the Drive mount when prompted.
5. Keep this tab open and your PC awake until the report prints.

> **If it stops or disconnects**, just **Run all** again — it resumes where it left off.

### Design notes
- Uses a **CPU runtime** (cheapest; GPU/TPU give no benefit).
- One **foreground cell** runs the whole extraction so Colab sees constant activity.
- All writes use an **atomic pattern** (`*.part` → verify → rename) so nothing is half-written.
- A small **state file** speeds up resume but correctness never depends on it.


In [ ]:
#@title 🔧 Configuration {display-mode: "form"}
#@markdown ### Source & destination
import os as _os

SOURCE_PATHS = _os.environ.get("SOURCE_PATHS", "/content/drive/MyDrive/ZIPS").split(";")  #@param {type:"raw"}
#@markdown Semicolon-separated list of Drive folders and/or individual .zip files.
#@markdown Default: `["/content/drive/MyDrive/ZIPS"]`

RECURSIVE = True  #@param {type:"boolean"}
ZIP_GLOB = "*.zip"  #@param {type:"string"}
#@markdown Case-insensitive glob pattern for zip files.

#@markdown ---
#@markdown ### Destination mode
DEST_MODE = "folder_next_to_zip"  #@param ["folder_next_to_zip", "same_folder", "custom"]
#@markdown - `folder_next_to_zip` — creates `<zipname>/` beside each zip (default)
#@markdown - `same_folder` — extracts into the same folder as the zip
#@markdown - `custom` — all zips merged into DEST_DIR

DEST_DIR = "/content/drive/MyDrive/Extracted"  #@param {type:"string"}
#@markdown Only used when DEST_MODE = "custom".

STRIP_SINGLE_TOP_FOLDER = True  #@param {type:"boolean"}
#@markdown Avoid double nesting (name/name/) when a zip has a single top folder.

#@markdown ---
#@markdown ### Behaviour
ON_EXISTING = "skip_if_same_size"  #@param ["skip_if_same_size", "overwrite", "keep_both"]
DELETE_ZIP_AFTER_SUCCESS = False  #@param {type:"boolean"}
EXTRACT_NESTED_ZIPS = False  #@param {type:"boolean"}
MAX_NESTING_DEPTH = 2  #@param {type:"integer"}

#@markdown ---
#@markdown ### Encryption / encoding
PASSWORD = ""  #@param {type:"string"}
#@markdown Leave blank; if an encrypted entry is found, the notebook asks via getpass.
ZIP_NAME_ENCODING = ""  #@param {type:"string"}
#@markdown For old non-UTF-8 zips: cp932, gbk, cp866, etc.

#@markdown ---
#@markdown ### Performance & limits
WORKERS = 4  #@param {type:"slider", min:1, max:8, step:1}
CHUNK_MB = 16  #@param {type:"integer"}
MAX_RUN_HOURS = 11  #@param {type:"number"}
#@markdown Free tier ≈ 12 h max; raise on paid plans.

#@markdown ---
#@markdown ### Flags
DRY_RUN = False  #@param {type:"boolean"}
IGNORE_SPACE_CHECK = True  #@param {type:"boolean"}
UNMOUNT_AT_END = True  #@param {type:"boolean"}
SAVE_REPORT = False  #@param {type:"boolean"}
KEEP_ALIVE_JS = False  #@param {type:"boolean"}

# ── Env-var overrides (for headless / local testing) ──
def _bool(v):
    return str(v).strip().lower() in ("1", "true", "yes")

if _os.environ.get("_GDUZ_TESTING"):
    SOURCE_PATHS   = _os.environ.get("SOURCE_PATHS", ";".join(SOURCE_PATHS)).split(";")
    RECURSIVE      = _bool(_os.environ.get("RECURSIVE", RECURSIVE))
    DEST_MODE      = _os.environ.get("DEST_MODE", DEST_MODE)
    DEST_DIR       = _os.environ.get("DEST_DIR", DEST_DIR)
    STRIP_SINGLE_TOP_FOLDER = _bool(_os.environ.get("STRIP_SINGLE_TOP_FOLDER", STRIP_SINGLE_TOP_FOLDER))
    ON_EXISTING    = _os.environ.get("ON_EXISTING", ON_EXISTING)
    DELETE_ZIP_AFTER_SUCCESS = _bool(_os.environ.get("DELETE_ZIP_AFTER_SUCCESS", DELETE_ZIP_AFTER_SUCCESS))
    EXTRACT_NESTED_ZIPS = _bool(_os.environ.get("EXTRACT_NESTED_ZIPS", EXTRACT_NESTED_ZIPS))
    MAX_NESTING_DEPTH = int(_os.environ.get("MAX_NESTING_DEPTH", MAX_NESTING_DEPTH))
    PASSWORD       = _os.environ.get("PASSWORD", PASSWORD)
    ZIP_NAME_ENCODING = _os.environ.get("ZIP_NAME_ENCODING", ZIP_NAME_ENCODING)
    WORKERS        = int(_os.environ.get("WORKERS", WORKERS))
    CHUNK_MB       = int(_os.environ.get("CHUNK_MB", CHUNK_MB))
    MAX_RUN_HOURS  = float(_os.environ.get("MAX_RUN_HOURS", MAX_RUN_HOURS))
    DRY_RUN        = _bool(_os.environ.get("DRY_RUN", DRY_RUN))
    IGNORE_SPACE_CHECK = _bool(_os.environ.get("IGNORE_SPACE_CHECK", IGNORE_SPACE_CHECK))
    UNMOUNT_AT_END = _bool(_os.environ.get("UNMOUNT_AT_END", UNMOUNT_AT_END))
    SAVE_REPORT    = _bool(_os.environ.get("SAVE_REPORT", SAVE_REPORT))
    KEEP_ALIVE_JS  = _bool(_os.environ.get("KEEP_ALIVE_JS", KEEP_ALIVE_JS))

WORKERS = max(1, min(8, WORKERS))
CHUNK_SIZE = CHUNK_MB * 1024 * 1024
print("✅ Configuration loaded.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 2 — Setup: imports, IN_COLAB detection, Drive mount, helpers
# ══════════════════════════════════════════════════════════════════════════
import sys, os, time, json, shutil, hashlib, struct, re, pathlib, fnmatch
import zipfile, io, logging, traceback, threading, ctypes, errno, getpass
import tempfile, datetime, math, random, signal, copy, csv
from collections import defaultdict, OrderedDict
from concurrent.futures import ThreadPoolExecutor, as_completed, Future
from typing import List, Dict, Tuple, Optional, Any, Set

try:
    from tqdm.auto import tqdm
except ImportError:
    os.system("pip install -q tqdm")
    from tqdm.auto import tqdm

# ── Colab detection ──
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

# ── Drive mount path ──
DRIVE_MOUNT = "/content/drive"
if not IN_COLAB:
    # Local testing: use env var or a temp folder
    DRIVE_MOUNT = os.environ.get("DRIVE_MOUNT", DRIVE_MOUNT)

# ── Logging ──
_log_fmt = logging.Formatter(
    "[%(asctime)s] %(levelname)-7s  %(message)s", datefmt="%H:%M:%S"
)
_log_handler = logging.StreamHandler(sys.stdout)
_log_handler.setFormatter(_log_fmt)
logger = logging.getLogger("gduz")
logger.handlers.clear()
logger.addHandler(_log_handler)
logger.setLevel(logging.INFO)
logger.propagate = False

# ── Human-readable sizes ──
def human(n: float, suffix="B") -> str:
    """Return human-readable byte string."""
    for unit in ("", "K", "M", "G", "T", "P"):
        if abs(n) < 1024:
            return f"{n:,.1f} {unit}{suffix}"
        n /= 1024
    return f"{n:,.1f} E{suffix}"

# ── Retry with exponential backoff + jitter ──
TRANSIENT_ERRNOS = {5, 107, 110, 112, 121}   # EIO, ENOTCONN, ETIMEDOUT, …

def is_transient(exc: Exception) -> bool:
    """Return True for errors that may self-heal after a wait."""
    if isinstance(exc, (ConnectionError, TimeoutError)):
        return True
    if isinstance(exc, OSError) and getattr(exc, "errno", None) in TRANSIENT_ERRNOS:
        return True
    msg = str(exc).lower()
    return "input/output error" in msg or "transport endpoint" in msg

_consecutive_errors = 0
_consecutive_lock = threading.Lock()

def reset_consecutive_errors():
    global _consecutive_errors
    with _consecutive_lock:
        _consecutive_errors = 0

def bump_consecutive_errors() -> int:
    global _consecutive_errors
    with _consecutive_lock:
        _consecutive_errors += 1
        return _consecutive_errors

def retry(fn, *args, max_retries=6, label="op", **kwargs):
    """Call fn(*args, **kwargs) with exponential backoff on transient errors."""
    delays = [2, 4, 8, 16, 32, 60]
    last_exc = None
    for attempt in range(max_retries + 1):
        try:
            result = fn(*args, **kwargs)
            reset_consecutive_errors()
            return result
        except Exception as exc:
            last_exc = exc
            if not is_transient(exc) or attempt == max_retries:
                raise
            n = bump_consecutive_errors()
            if n >= 20:
                raise RuntimeError(
                    f"Too many consecutive transient errors ({n}). "
                    "Drive quota may be exhausted. Wait ~24 h, or if the zip "
                    "is a shared file, copy it into your own Drive and point "
                    "SOURCE_PATHS at the copy. If Drive storage is full, free "
                    "space and empty the Trash."
                ) from exc
            delay = delays[min(attempt, len(delays) - 1)]
            jitter = random.uniform(0, delay * 0.3)
            logger.warning(
                "%s failed (%s), retry %d/%d in %.1f s …",
                label, exc, attempt + 1, max_retries, delay + jitter,
            )
            time.sleep(delay + jitter)
    raise last_exc  # unreachable

# ── Safe path helpers ──
def safe_name(name: str) -> str:
    """Sanitise a zip entry name: block zip-slip, strip drive letters, fix separators."""
    # Treat backslash as separator
    name = name.replace("\\", "/")
    # Strip drive letters (C:, D:, etc.)
    if len(name) >= 2 and name[1] == ":":
        name = name[2:]
    # Remove leading slashes
    name = name.lstrip("/")
    # Reject .. components
    parts = name.split("/")
    clean_parts = []
    for p in parts:
        if p in ("", "."):
            continue
        if p == "..":
            logger.warning("Zip-slip blocked: component '..' in %r", name)
            continue
        # Truncate individual components to 255 bytes
        encoded = p.encode("utf-8", errors="replace")
        if len(encoded) > 255:
            h = hashlib.sha256(encoded).hexdigest()[:12]
            p = encoded[:200].decode("utf-8", errors="replace") + f"__{h}"
            logger.info("Name truncated: … → %s", p)
        clean_parts.append(p)
    return "/".join(clean_parts)


def decode_entry_name(info: zipfile.ZipInfo, encoding: str = "") -> str:
    """Recover the real filename from a ZipInfo, handling legacy encodings."""
    # If the UTF-8 flag is set, Python already decoded correctly
    if info.flag_bits & 0x800:
        return info.filename
    # Otherwise, get the raw cp437 bytes and try UTF-8, then user encoding
    try:
        raw = info.filename.encode("cp437")
    except (UnicodeDecodeError, UnicodeEncodeError):
        return info.filename
    for enc in ("utf-8", encoding) if encoding else ("utf-8",):
        if not enc:
            continue
        try:
            return raw.decode(enc)
        except (UnicodeDecodeError, LookupError):
            continue
    return info.filename  # fall back

# ── Atomic JSON write ──
def atomic_json_write(path: str, data: Any):
    """Write JSON atomically: write .tmp → flush/fsync → rename."""
    tmp = path + ".tmp"
    try:
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=1, default=str)
            f.flush()
            try:
                os.fsync(f.fileno())
            except OSError:
                pass
        os.replace(tmp, path)
    except Exception:
        try:
            os.unlink(tmp)
        except OSError:
            pass
        raise

def atomic_json_read(path: str, default=None):
    """Read JSON, return default on any error."""
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return default

# ── Drive mount with retry ──
def mount_drive(force=False):
    """Mount Google Drive (Colab only)."""
    if not IN_COLAB:
        logger.info("Not in Colab — skipping Drive mount (DRIVE_MOUNT=%s)", DRIVE_MOUNT)
        return
    from google.colab import drive
    for attempt in range(3):
        try:
            drive.mount(DRIVE_MOUNT, force_remount=force)
            logger.info("Drive mounted at %s", DRIVE_MOUNT)
            return
        except Exception as exc:
            if attempt < 2:
                logger.warning("Mount failed (%s), retrying in 5 s …", exc)
                time.sleep(5)
            else:
                raise RuntimeError(
                    "Could not mount Google Drive after 3 attempts. "
                    "Check your connection and try again."
                ) from exc

mount_drive()

# ── Disk helpers ──
def free_drive_bytes() -> int:
    """Free bytes on the Drive FUSE mount."""
    try:
        return shutil.disk_usage(DRIVE_MOUNT).free
    except Exception:
        return 0

def free_local_bytes() -> int:
    """Free bytes on the local (VM) disk."""
    try:
        return shutil.disk_usage("/").free
    except Exception:
        try:
            return shutil.disk_usage(os.getcwd()).free
        except Exception:
            return 0

# ── OS sync helper ──
def os_sync():
    """Best-effort os.sync()."""
    try:
        if hasattr(os, "sync"):
            os.sync()
    except Exception:
        pass

# ── State file location ──
STATE_DIR = os.path.join(DRIVE_MOUNT, "MyDrive", ".gduz_state")
STATE_FILE = os.path.join(STATE_DIR, "state.json")
if os.environ.get("_GDUZ_STATE_DIR"):
    STATE_DIR = os.environ["_GDUZ_STATE_DIR"]
    STATE_FILE = os.path.join(STATE_DIR, "state.json")

# ── Timing ──
RUN_START = time.monotonic()
DEADLINE = RUN_START + MAX_RUN_HOURS * 3600

def time_left() -> float:
    return DEADLINE - time.monotonic()

def past_deadline() -> bool:
    return time.monotonic() >= DEADLINE

print(f"✅ Setup done.  IN_COLAB={IN_COLAB}  DRIVE_MOUNT={DRIVE_MOUNT}")
print(f"   Free Drive: {human(free_drive_bytes())}   Free local: {human(free_local_bytes())}")
print(f"   Time budget: {MAX_RUN_HOURS} h  (deadline in {time_left()/3600:.1f} h)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 3 — Optional keep-alive JavaScript
# ══════════════════════════════════════════════════════════════════════════
#
# ⚠️  UNOFFICIAL — may stop working at any time.
# Colab restricts actions that bypass its idle-detection policies.
# The REAL protection against disconnects is resume (cells 5-6).
# Use at your own risk.

if KEEP_ALIVE_JS and IN_COLAB:
    from IPython.display import display, Javascript
    display(Javascript("""
    (function keepAlive() {
        try {
            // Click the "connect" button if it appears
            var btn = document.querySelector("colab-connect-button");
            if (btn) btn.click();
        } catch(e) {}
        setTimeout(keepAlive, 60000);
    })();
    console.log("keep-alive JS started (clicks connect button every 60 s)");
    """))
    print("⚡ Keep-alive JS injected (unofficial, best-effort).")
else:
    print("ℹ️  Keep-alive JS disabled (KEEP_ALIVE_JS=False or not in Colab).")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 4 — Scan + Pre-flight  (read-only, writes nothing)
# ══════════════════════════════════════════════════════════════════════════

class ZipMeta:
    """Metadata for one zip file."""
    __slots__ = (
        "path", "size", "entry_count", "total_uncompressed",
        "largest_entry", "compression_methods", "encrypted",
        "bad_names", "max_children", "dest_dir", "state",
        "error", "entries", "has_single_top_folder",
        "needs_pyzipper", "needs_7z", "is_split",
    )
    def __init__(self, path: str):
        self.path = path
        self.size = 0
        self.entry_count = 0
        self.total_uncompressed = 0
        self.largest_entry = 0
        self.compression_methods: Set[int] = set()
        self.encrypted = False
        self.bad_names: List[str] = []
        self.max_children = 0
        self.dest_dir = ""
        self.state = "new"       # new | partly_done | done
        self.error = ""
        self.entries: List[zipfile.ZipInfo] = []
        self.has_single_top_folder = False
        self.needs_pyzipper = False
        self.needs_7z = False
        self.is_split = False

# ── Discover zip files ──
def discover_zips(source_paths: List[str], recursive: bool, glob_pat: str) -> List[str]:
    """Walk source paths and return deduplicated list of zip files."""
    found = []
    seen_real = set()
    glob_lower = glob_pat.lower()

    def _matches(name):
        return fnmatch.fnmatch(name.lower(), glob_lower)

    for src in source_paths:
        src = src.strip()
        if not src:
            continue
        if not os.path.exists(src):
            logger.error("SOURCE_PATH does not exist: %s", src)
            continue
        if os.path.isfile(src):
            rp = os.path.realpath(src)
            if rp not in seen_real and _matches(os.path.basename(src)):
                seen_real.add(rp)
                found.append(src)
            continue
        # Directory
        if recursive:
            for root, dirs, files in os.walk(src, followlinks=False):
                # Skip hidden directories
                dirs[:] = [d for d in dirs if not d.startswith(".")]
                for f in files:
                    if _matches(f):
                        fp = os.path.join(root, f)
                        rp = os.path.realpath(fp)
                        if rp not in seen_real:
                            seen_real.add(rp)
                            found.append(fp)
        else:
            try:
                with os.scandir(src) as it:
                    for entry in it:
                        if entry.is_file(follow_symlinks=False) and _matches(entry.name):
                            rp = os.path.realpath(entry.path)
                            if rp not in seen_real:
                                seen_real.add(rp)
                                found.append(entry.path)
            except OSError as exc:
                logger.error("Cannot scan %s: %s", src, exc)
    return sorted(found)

# ── Analyse one zip ──
def analyse_zip(path: str) -> ZipMeta:
    """Read only the central directory; collect metadata."""
    meta = ZipMeta(path)
    meta.size = os.path.getsize(path)

    # Check for split archive markers
    base = os.path.splitext(path)[0]
    for ext in (".z01", ".zip.001"):
        if os.path.exists(base + ext):
            meta.is_split = True
            break

    try:
        with open(path, "rb", buffering=CHUNK_SIZE) as raw:
            with zipfile.ZipFile(raw) as zf:
                infos = zf.infolist()
                meta.entries = infos
                meta.entry_count = len(infos)
                children_count = defaultdict(int)  # parent_dir -> count
                top_folders = set()

                for info in infos:
                    name = decode_entry_name(info, ZIP_NAME_ENCODING)
                    sname = safe_name(name)

                    if sname != name.replace("\\", "/").lstrip("/"):
                        meta.bad_names.append(name)

                    meta.total_uncompressed += info.file_size
                    if info.file_size > meta.largest_entry:
                        meta.largest_entry = info.file_size

                    meta.compression_methods.add(info.compress_type)

                    if info.flag_bits & 0x1:
                        meta.encrypted = True

                    # Count direct children per folder
                    parts = sname.rstrip("/").split("/")
                    if len(parts) >= 1:
                        top_folders.add(parts[0])
                    parent = "/".join(parts[:-1]) if len(parts) > 1 else ""
                    children_count[parent] += 1

                if children_count:
                    meta.max_children = max(children_count.values())

                # Single top folder detection
                if len(top_folders) == 1:
                    meta.has_single_top_folder = True

    except zipfile.BadZipFile as exc:
        meta.error = f"BadZipFile: {exc}"
    except Exception as exc:
        meta.error = f"Error reading: {exc}"

    # Check for Deflate64 (method 9) or AES
    DEFLATE64 = 9
    AES_METHOD = 99
    if DEFLATE64 in meta.compression_methods:
        meta.needs_7z = True
    if AES_METHOD in meta.compression_methods:
        meta.needs_pyzipper = True

    return meta

# ── Compute destination directory ──
def compute_dest(meta: ZipMeta) -> str:
    """Determine the extraction destination folder."""
    zip_dir = os.path.dirname(meta.path)
    zip_stem = os.path.splitext(os.path.basename(meta.path))[0]

    if DEST_MODE == "same_folder":
        return zip_dir
    elif DEST_MODE == "custom":
        return DEST_DIR
    else:  # folder_next_to_zip
        return os.path.join(zip_dir, zip_stem)

# ── Check existing state ──
def check_zip_state(meta: ZipMeta, state_data: dict) -> str:
    """Determine if a zip is new, partly done, or fully done."""
    dest = meta.dest_dir
    zip_key = os.path.realpath(meta.path)

    # If state file says done, still verify a sample
    if zip_key in state_data.get("done_zips", {}):
        return "done"

    if not os.path.isdir(dest):
        return "new"

    # Check if any extracted files exist
    done_count = 0
    total_files = 0
    for info in meta.entries:
        if info.is_dir():
            continue
        total_files += 1
        name = decode_entry_name(info, ZIP_NAME_ENCODING)
        sname = safe_name(name)
        if not sname:
            continue

        # Apply STRIP_SINGLE_TOP_FOLDER
        out_name = sname
        if STRIP_SINGLE_TOP_FOLDER and meta.has_single_top_folder and DEST_MODE != "same_folder":
            parts = out_name.split("/", 1)
            if len(parts) > 1:
                out_name = parts[1]
            else:
                out_name = parts[0]

        out_path = os.path.join(dest, out_name)
        if os.path.exists(out_path):
            try:
                if os.path.getsize(out_path) == info.file_size:
                    done_count += 1
            except OSError:
                pass

    if total_files == 0:
        return "done"
    if done_count == total_files:
        return "done"
    if done_count > 0:
        return "partly_done"
    return "new"


# ═══════════════════  RUN DISCOVERY  ═══════════════════

zip_paths = discover_zips(SOURCE_PATHS, RECURSIVE, ZIP_GLOB)

_GDUZ_STOP = False
_GDUZ_STOP_REASON = ""

if not zip_paths:
    print("❌ No zip files found in SOURCE_PATHS:")
    for sp in SOURCE_PATHS:
        print(f"   {sp}  (exists={os.path.exists(sp.strip())})")
    _GDUZ_STOP = True
    _GDUZ_STOP_REASON = "No zip files found"

# Initialize variables that later cells need even if we stop early
zip_metas: List[ZipMeta] = []
total_uncompressed = 0
bytes_to_extract = 0
state_data = atomic_json_read(STATE_FILE, {"done_zips": {}})
warnings_list = []
errors_list = []

if not _GDUZ_STOP:
    print(f"📂 Found {len(zip_paths)} zip file(s). Analysing …\n")

    for zp in tqdm(zip_paths, desc="Analysing", unit="zip"):
        meta = analyse_zip(zp)
        meta.dest_dir = compute_dest(meta)
        meta.state = check_zip_state(meta, state_data)
        zip_metas.append(meta)
        total_uncompressed += meta.total_uncompressed
        if meta.state != "done":
            bytes_to_extract += meta.total_uncompressed

    # ── Pre-flight checks ──
    free_drive = free_drive_bytes()
    free_local = free_local_bytes()

    print("\n" + "=" * 90)
    print(f"{'ZIP FILE':<40} {'SIZE':>10} {'ENTRIES':>8} {'UNCOMPRESSED':>14} {'STATE':>12}")
    print("-" * 90)
    for m in zip_metas:
        state_icon = {"new": "🆕", "partly_done": "🔄", "done": "✅"}.get(m.state, "❓")
        name = os.path.basename(m.path)
        if len(name) > 38:
            name = name[:35] + "…"
        err = f" ⚠ {m.error}" if m.error else ""
        print(f"{name:<40} {human(m.size):>10} {m.entry_count:>8} {human(m.total_uncompressed):>14} {state_icon:>2} {m.state}{err}")
    print("=" * 90)
    print(f"Total: {len(zip_metas)} zips, {human(total_uncompressed)} uncompressed, "
          f"{human(bytes_to_extract)} still to extract")
    print(f"Free Drive: ~{human(free_drive)}   Free local: ~{human(free_local)}\n")

    # Warnings
    for m in zip_metas:
        if m.max_children > 9000:
            w = f"⚠️  {os.path.basename(m.path)}: a folder would get {m.max_children:,} direct entries (>9000) — may cause Drive I/O errors"
            warnings_list.append(w)
            print(w)
        if m.needs_7z:
            w = f"⚠️  {os.path.basename(m.path)}: contains Deflate64 entries — will try 7z fallback"
            warnings_list.append(w)
            print(w)
        if m.needs_pyzipper:
            w = f"⚠️  {os.path.basename(m.path)}: contains AES-encrypted entries — will try pyzipper"
            warnings_list.append(w)
            print(w)
        if m.encrypted:
            w = f"⚠️  {os.path.basename(m.path)}: encrypted entries detected"
            warnings_list.append(w)
            print(w)
        if m.bad_names:
            w = f"⚠️  {os.path.basename(m.path)}: {len(m.bad_names)} entries with unusual names will be renamed"
            warnings_list.append(w)
            print(w)
        if m.is_split:
            w = f"⚠️  {os.path.basename(m.path)}: appears to be part of a split archive — extraction may fail"
            warnings_list.append(w)
            print(w)
        if m.error:
            errors_list.append(f"❌  {os.path.basename(m.path)}: {m.error}")

    # Space check
    if bytes_to_extract > 0 and not IGNORE_SPACE_CHECK:
        if free_drive > 0 and bytes_to_extract > free_drive * 0.95:
            msg = (f"❌ Not enough Drive space: need ~{human(bytes_to_extract)}, "
                   f"free ~{human(free_drive)}. Free space or set IGNORE_SPACE_CHECK=True.")
            errors_list.append(msg)
            print(msg)

    for e in errors_list:
        print(e)

    if errors_list and not IGNORE_SPACE_CHECK:
        _GDUZ_STOP = True
        _GDUZ_STOP_REASON = "Pre-flight errors found. Fix them and re-run."
        print(f"❌ {_GDUZ_STOP_REASON}")

    if DRY_RUN:
        print("\n🏁 DRY_RUN=True — stopping here. No files were written.")
        _GDUZ_STOP = True
        _GDUZ_STOP_REASON = "DRY_RUN"

    if not _GDUZ_STOP:
        if bytes_to_extract == 0:
            print("\n✅ All zips are already fully extracted. Nothing to do.")
        else:
            n_todo = sum(1 for m in zip_metas if m.state != "done")
            print(f"\n🚀 {n_todo} zip(s) to process ({human(bytes_to_extract)} to extract). Proceeding to extraction …")
else:
    print(f"⚠️ Stopped: {_GDUZ_STOP_REASON}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 5 — MAIN RUN: Extraction engine
# ══════════════════════════════════════════════════════════════════════════
import subprocess

# ── Password handling ──
_zip_password: Optional[bytes] = None
if PASSWORD:
    _zip_password = PASSWORD.encode("utf-8")

def get_password() -> bytes:
    """Get the zip password, asking interactively if needed."""
    global _zip_password
    if _zip_password is None:
        pw = getpass.getpass("🔑 Enter zip password: ")
        _zip_password = pw.encode("utf-8")
    return _zip_password

# ── Fallback tool installation ──
_pyzipper_available = None
_7z_available = None

def ensure_pyzipper():
    global _pyzipper_available
    if _pyzipper_available is not None:
        return _pyzipper_available
    try:
        import pyzipper  # noqa: F401
        _pyzipper_available = True
    except ImportError:
        logger.info("Installing pyzipper …")
        os.system("pip install -q pyzipper")
        try:
            import pyzipper  # noqa: F401
            _pyzipper_available = True
        except ImportError:
            _pyzipper_available = False
    return _pyzipper_available

def ensure_7z():
    global _7z_available
    if _7z_available is not None:
        return _7z_available
    if shutil.which("7z"):
        _7z_available = True
        return True
    logger.info("Installing p7zip-full …")
    ret = os.system("apt-get install -y -qq p7zip-full 2>/dev/null")
    _7z_available = shutil.which("7z") is not None
    return _7z_available

# ── Stats tracking ──
class RunStats:
    def __init__(self):
        self.lock = threading.Lock()
        self.files_done = 0
        self.files_skipped = 0
        self.bytes_done = 0
        self.files_failed = 0
        self.files_renamed = 0
        self.dirs_created = 0
        self.zips_done = 0
        self.zips_failed = 0
        self.zip_results: Dict[str, dict] = OrderedDict()
        self.failures: List[dict] = []
        self.part_files_created: List[str] = []  # track for cleanup
        self.skipped_entries: List[dict] = []
        self.renamed_entries: List[dict] = []
        self.nested_zips_found: List[str] = []

    def add_part(self, path: str):
        with self.lock:
            self.part_files_created.append(path)

    def remove_part(self, path: str):
        with self.lock:
            if path in self.part_files_created:
                self.part_files_created.remove(path)

stats = RunStats()

# ── Adaptive throttle ──
class AdaptiveThrottle:
    def __init__(self, max_workers: int):
        self.max_workers = max_workers
        self.current_workers = max_workers
        self.lock = threading.Lock()
        self.error_count = 0
        self.success_count = 0
        self.throttle_delay = 0.0

    def on_error(self):
        with self.lock:
            self.error_count += 1
            self.success_count = 0
            self.current_workers = max(1, self.current_workers - 1)
            self.throttle_delay = min(5.0, self.throttle_delay + 0.5)

    def on_success(self):
        with self.lock:
            self.success_count += 1
            self.error_count = 0
            if self.success_count > 20 and self.current_workers < self.max_workers:
                self.current_workers = min(self.max_workers, self.current_workers + 1)
                self.success_count = 0
            if self.success_count > 10:
                self.throttle_delay = max(0, self.throttle_delay - 0.1)

    def wait(self):
        d = self.throttle_delay
        if d > 0:
            time.sleep(d)

throttle = AdaptiveThrottle(WORKERS)

# ── Local disk guard ──
def check_local_disk():
    """Return True if local disk is OK, False if critically low."""
    free = free_local_bytes()
    total = 1
    try:
        total = shutil.disk_usage("/").total
    except Exception:
        try:
            total = shutil.disk_usage(os.getcwd()).total
        except Exception:
            return True
    pct_free = free / total if total else 0
    if free < 2 * 1024**3 or pct_free < 0.03:  # <2 GB or <3%
        logger.warning("Local disk critically low: %s free (%.1f%%). Syncing …", human(free), pct_free * 100)
        os_sync()
        time.sleep(10)
        free2 = free_local_bytes()
        if free2 < 1 * 1024**3:
            return False
    return True

# ── Extract a single member ──
def extract_member(
    zip_path: str, info: zipfile.ZipInfo, dest_dir: str,
    strip_top: bool, top_folder: str, meta: ZipMeta,
    pbar: tqdm, password: Optional[bytes] = None
) -> dict:
    """Extract one member from a zip file. Returns result dict."""
    name = decode_entry_name(info, ZIP_NAME_ENCODING)
    sname = safe_name(name)
    result = {"name": name, "safe_name": sname, "status": "ok", "error": ""}

    if not sname:
        result["status"] = "skipped"
        result["error"] = "empty name after sanitisation"
        with stats.lock:
            stats.skipped_entries.append({"zip": zip_path, "name": name, "reason": "empty name"})
        return result

    # Track renames
    original_name = name.replace("\\", "/").lstrip("/")
    if sname != original_name:
        result["renamed"] = True
        with stats.lock:
            stats.files_renamed += 1
            stats.renamed_entries.append({"zip": zip_path, "original": name, "safe": sname})

    # Strip single top folder
    out_name = sname
    if strip_top and top_folder:
        parts = out_name.split("/", 1)
        if len(parts) > 1:
            out_name = parts[1]
        # If it's just the top folder entry itself, skip (it's a dir)
        elif info.is_dir():
            result["status"] = "skipped"
            result["error"] = "stripped top folder dir"
            return result

    out_path = os.path.join(dest_dir, out_name)

    # Security: ensure out_path is within dest_dir
    real_dest = os.path.realpath(dest_dir)
    real_out = os.path.realpath(out_path)
    if not real_out.startswith(real_dest + os.sep) and real_out != real_dest:
        result["status"] = "blocked"
        result["error"] = f"zip-slip: {out_name} resolves outside destination"
        with stats.lock:
            stats.failures.append({"zip": zip_path, "name": name, "error": result["error"]})
        logger.warning("🚫 Zip-slip blocked: %s → %s", name, real_out)
        return result

    # Skip symlinks and special entries
    # External attributes: check for symlink (Unix mode has S_IFLNK = 0o120000)
    unix_attr = info.external_attr >> 16
    if unix_attr != 0 and (unix_attr & 0o170000) == 0o120000:
        result["status"] = "skipped"
        result["error"] = "symlink"
        with stats.lock:
            stats.skipped_entries.append({"zip": zip_path, "name": name, "reason": "symlink"})
        logger.info("Skipping symlink: %s", name)
        return result

    # Directory entry
    if info.is_dir():
        try:
            retry(os.makedirs, out_path, exist_ok=True, label=f"mkdir {out_name}")
            with stats.lock:
                stats.dirs_created += 1
        except Exception as exc:
            result["status"] = "error"
            result["error"] = str(exc)
        return result

    # File entry — check for resume
    if os.path.exists(out_path):
        try:
            existing_size = os.path.getsize(out_path)
        except OSError:
            existing_size = -1

        if ON_EXISTING == "skip_if_same_size" and existing_size == info.file_size:
            result["status"] = "skipped_existing"
            with stats.lock:
                stats.files_skipped += 1
                stats.bytes_done += info.file_size
            pbar.update(info.file_size)
            return result
        elif ON_EXISTING == "skip_if_same_size" and existing_size != info.file_size:
            logger.info("Size mismatch for %s: existing=%s, expected=%s — overwriting",
                        out_name, existing_size, info.file_size)
        elif ON_EXISTING == "keep_both":
            base, ext = os.path.splitext(out_path)
            counter = 1
            while os.path.exists(out_path):
                out_path = f"{base}_{counter}{ext}"
                counter += 1

    # Create parent dirs
    parent = os.path.dirname(out_path)
    if parent and not os.path.isdir(parent):
        retry(os.makedirs, parent, exist_ok=True, label=f"mkdir {parent}")

    # Extract with streaming
    part_path = out_path + ".part"

    # Delete stale .part file
    if os.path.exists(part_path):
        try:
            os.unlink(part_path)
        except OSError:
            pass

    stats.add_part(part_path)

    try:
        throttle.wait()

        # Each worker opens its own ZipFile handle
        with open(zip_path, "rb", buffering=CHUNK_SIZE) as raw_fh:
            with zipfile.ZipFile(raw_fh) as zf:
                try:
                    src = zf.open(info, pwd=password)
                except RuntimeError as exc:
                    if "password" in str(exc).lower() or "encrypted" in str(exc).lower():
                        pw = get_password()
                        src = zf.open(info, pwd=pw)
                    else:
                        raise

                with src:
                    with open(part_path, "wb") as dst:
                        written = 0
                        while True:
                            chunk = retry(src.read, CHUNK_SIZE, label=f"read {out_name}")
                            if not chunk:
                                break
                            retry(dst.write, chunk, label=f"write {out_name}")
                            written += len(chunk)
                            pbar.update(len(chunk))
                            with stats.lock:
                                stats.bytes_done += len(chunk)

                        dst.flush()
                        try:
                            os.fsync(dst.fileno())
                        except OSError:
                            pass

        # Verify size
        actual_size = os.path.getsize(part_path)
        if actual_size != info.file_size:
            raise RuntimeError(
                f"Size mismatch: wrote {actual_size}, expected {info.file_size}"
            )

        # Atomic rename
        os.replace(part_path, out_path)
        stats.remove_part(part_path)

        # Set mtime from zip
        try:
            mtime = time.mktime(info.date_time + (0, 0, -1))
            os.utime(out_path, (mtime, mtime))
        except Exception:
            pass

        with stats.lock:
            stats.files_done += 1

        throttle.on_success()
        result["status"] = "ok"

    except Exception as exc:
        result["status"] = "error"
        result["error"] = str(exc)
        with stats.lock:
            stats.files_failed += 1
            stats.failures.append({"zip": zip_path, "name": name, "error": str(exc)})
        throttle.on_error()
        logger.error("Failed: %s/%s — %s", os.path.basename(zip_path), name, exc)

        # Clean up .part file on error
        try:
            if os.path.exists(part_path):
                os.unlink(part_path)
        except OSError:
            pass
        stats.remove_part(part_path)

    return result


# ── 7z fallback extraction ──
def extract_with_7z(meta: ZipMeta, dest_dir: str) -> bool:
    """Extract a zip using 7z as a fallback. Returns True on success."""
    if not ensure_7z():
        logger.error("7z not available, cannot extract %s", meta.path)
        return False

    logger.info("Using 7z fallback for %s", os.path.basename(meta.path))
    os.makedirs(dest_dir, exist_ok=True)

    cmd = ["7z", "x", "-y", f"-o{dest_dir}", meta.path]
    if _zip_password:
        cmd.insert(3, f"-p{_zip_password.decode('utf-8', errors='replace')}")

    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=MAX_RUN_HOURS * 3600)
        if proc.returncode != 0:
            logger.error("7z failed (rc=%d): %s", proc.returncode, proc.stderr[:500])
            return False
        logger.info("7z extraction completed for %s", os.path.basename(meta.path))
        return True
    except subprocess.TimeoutExpired:
        logger.error("7z timed out for %s", meta.path)
        return False
    except Exception as exc:
        logger.error("7z error for %s: %s", meta.path, exc)
        return False


# ── pyzipper fallback ──
def extract_with_pyzipper(meta: ZipMeta, dest_dir: str, pbar: tqdm) -> bool:
    """Extract AES-encrypted zip using pyzipper. Returns True on success."""
    if not ensure_pyzipper():
        logger.error("pyzipper not available, cannot extract %s", meta.path)
        return False

    import pyzipper
    logger.info("Using pyzipper (AES) for %s", os.path.basename(meta.path))
    os.makedirs(dest_dir, exist_ok=True)

    password = _zip_password
    success = True

    try:
        with pyzipper.AESZipFile(meta.path, "r") as zf:
            for info in zf.infolist():
                if past_deadline():
                    logger.warning("Time budget exceeded during pyzipper extraction")
                    return False

                name = info.filename
                sname = safe_name(name)
                if not sname:
                    continue

                out_name = sname
                if STRIP_SINGLE_TOP_FOLDER and meta.has_single_top_folder and DEST_MODE != "same_folder":
                    parts = out_name.split("/", 1)
                    if len(parts) > 1:
                        out_name = parts[1]
                    elif info.is_dir():
                        continue

                out_path = os.path.join(dest_dir, out_name)

                if info.is_dir():
                    os.makedirs(out_path, exist_ok=True)
                    continue

                # Skip if already done
                if os.path.exists(out_path) and ON_EXISTING == "skip_if_same_size":
                    try:
                        if os.path.getsize(out_path) == info.file_size:
                            pbar.update(info.file_size)
                            continue
                    except OSError:
                        pass

                parent = os.path.dirname(out_path)
                if parent:
                    os.makedirs(parent, exist_ok=True)

                part_path = out_path + ".part"
                stats.add_part(part_path)

                try:
                    if not password and info.flag_bits & 0x1:
                        password = get_password()
                    data = zf.read(info.filename, pwd=password)
                    with open(part_path, "wb") as f:
                        f.write(data)
                        f.flush()
                        try:
                            os.fsync(f.fileno())
                        except OSError:
                            pass

                    if os.path.getsize(part_path) != info.file_size:
                        raise RuntimeError("Size mismatch after pyzipper extract")

                    os.replace(part_path, out_path)
                    stats.remove_part(part_path)
                    pbar.update(info.file_size)
                    with stats.lock:
                        stats.files_done += 1
                        stats.bytes_done += info.file_size

                except Exception as exc:
                    logger.error("pyzipper failed on %s: %s", name, exc)
                    success = False
                    with stats.lock:
                        stats.failures.append({"zip": meta.path, "name": name, "error": str(exc)})
                    try:
                        if os.path.exists(part_path):
                            os.unlink(part_path)
                    except OSError:
                        pass
                    stats.remove_part(part_path)

    except Exception as exc:
        logger.error("pyzipper failed for %s: %s", meta.path, exc)
        return False

    return success


# ── Process one zip file ──
def process_zip(meta: ZipMeta, pbar_total: tqdm) -> dict:
    """Extract all members of one zip. Returns result dict."""
    zip_name = os.path.basename(meta.path)
    result = {
        "zip": meta.path,
        "name": zip_name,
        "entries": meta.entry_count,
        "size": meta.size,
        "uncompressed": meta.total_uncompressed,
        "status": "ok",
        "error": "",
        "time_s": 0,
        "files_done": 0,
        "files_skipped": 0,
        "files_failed": 0,
    }
    t0 = time.monotonic()

    if meta.state == "done":
        result["status"] = "already_done"
        result["files_skipped"] = meta.entry_count
        pbar_total.update(meta.total_uncompressed)
        logger.info("✅ %s — already fully extracted, skipping", zip_name)
        return result

    dest_dir = meta.dest_dir
    logger.info("📦 Processing: %s → %s", zip_name, dest_dir)

    # Determine strip settings
    strip_top = (STRIP_SINGLE_TOP_FOLDER and meta.has_single_top_folder
                 and DEST_MODE != "same_folder")
    top_folder = ""
    if strip_top and meta.entries:
        top_folder = safe_name(decode_entry_name(meta.entries[0], ZIP_NAME_ENCODING)).split("/")[0]

    # Create destination
    retry(os.makedirs, dest_dir, exist_ok=True, label=f"mkdir {dest_dir}")

    # ── Fallback paths ──
    if meta.needs_7z and not meta.needs_pyzipper:
        # Deflate64: must use 7z
        ok = extract_with_7z(meta, dest_dir)
        result["time_s"] = time.monotonic() - t0
        result["status"] = "ok_7z" if ok else "error_7z"
        if not ok:
            result["error"] = "7z extraction failed"
            with stats.lock:
                stats.zips_failed += 1
        else:
            with stats.lock:
                stats.zips_done += 1
        pbar_total.update(meta.total_uncompressed)
        return result

    if meta.needs_pyzipper:
        # AES: try pyzipper
        pbar_zip = tqdm(total=meta.total_uncompressed, unit="B", unit_scale=True,
                        desc=zip_name[:30], leave=False)
        ok = extract_with_pyzipper(meta, dest_dir, pbar_zip)
        pbar_zip.close()
        pbar_total.update(meta.total_uncompressed)
        result["time_s"] = time.monotonic() - t0
        result["status"] = "ok_pyzipper" if ok else "error_pyzipper"
        if not ok:
            result["error"] = "pyzipper extraction failed"
            with stats.lock:
                stats.zips_failed += 1
        else:
            with stats.lock:
                stats.zips_done += 1
        return result

    if meta.error:
        # Cannot read with zipfile at all
        logger.warning("Cannot read %s with zipfile: %s — trying 7z", zip_name, meta.error)
        if ensure_7z():
            ok = extract_with_7z(meta, dest_dir)
            result["time_s"] = time.monotonic() - t0
            result["status"] = "ok_7z" if ok else "error"
            if not ok:
                result["error"] = f"Original: {meta.error}; 7z also failed"
                with stats.lock:
                    stats.zips_failed += 1
            else:
                with stats.lock:
                    stats.zips_done += 1
            pbar_total.update(meta.total_uncompressed)
            return result
        else:
            result["error"] = meta.error
            result["status"] = "error"
            with stats.lock:
                stats.zips_failed += 1
            pbar_total.update(meta.total_uncompressed)
            return result

    # ── Standard zipfile extraction with ThreadPoolExecutor ──
    file_entries = [info for info in meta.entries if not info.is_dir()]
    dir_entries = [info for info in meta.entries if info.is_dir()]

    # Create directories first
    for info in dir_entries:
        dname = safe_name(decode_entry_name(info, ZIP_NAME_ENCODING))
        if not dname:
            continue
        out_name = dname
        if strip_top and top_folder:
            parts = out_name.split("/", 1)
            if len(parts) > 1:
                out_name = parts[1]
            else:
                continue
        dp = os.path.join(dest_dir, out_name)
        try:
            retry(os.makedirs, dp, exist_ok=True, label=f"mkdir")
        except Exception as exc:
            logger.warning("Could not create dir %s: %s", dp, exc)

    # Password detection
    password = _zip_password
    if meta.encrypted and not password:
        password = get_password()

    pbar_zip = tqdm(total=meta.total_uncompressed, unit="B", unit_scale=True,
                    desc=zip_name[:30], leave=False)

    zip_files_done = 0
    zip_files_skipped = 0
    zip_files_failed = 0
    last_heartbeat = time.monotonic()
    last_state_save = time.monotonic()
    stop_early = False

    effective_workers = min(WORKERS, max(1, len(file_entries)))

    if effective_workers == 1 or len(file_entries) <= 2:
        # Sequential mode — simpler, better for Drive I/O
        for info in file_entries:
            if past_deadline():
                logger.warning("⏰ Time budget exceeded. Stopping after current file.")
                stop_early = True
                break
            if not check_local_disk():
                logger.error("💾 Local disk critically low. Stopping.")
                stop_early = True
                break

            r = extract_member(meta.path, info, dest_dir, strip_top, top_folder,
                               meta, pbar_zip, password)
            if r["status"] == "ok":
                zip_files_done += 1
            elif r["status"].startswith("skip"):
                zip_files_skipped += 1
            elif r["status"] in ("error", "blocked"):
                zip_files_failed += 1

            # Heartbeat
            now = time.monotonic()
            if now - last_heartbeat > 30:
                elapsed = now - RUN_START
                speed = stats.bytes_done / elapsed if elapsed > 0 else 0
                eta_s = (pbar_total.total - stats.bytes_done) / speed if speed > 0 else 0
                print(f"  💓 {stats.files_done + stats.files_skipped} files | "
                      f"{human(stats.bytes_done)}/{human(pbar_total.total)} | "
                      f"{human(speed)}/s | ETA {eta_s/60:.0f} min | "
                      f"disk {human(free_local_bytes())}")
                last_heartbeat = now

            # State save
            if now - last_state_save > 30:
                try:
                    os.makedirs(STATE_DIR, exist_ok=True)
                    state_data["last_update"] = datetime.datetime.now().isoformat()
                    atomic_json_write(STATE_FILE, state_data)
                except Exception:
                    pass
                last_state_save = now
    else:
        # Parallel mode
        with ThreadPoolExecutor(max_workers=effective_workers) as pool:
            futures: Dict[Future, zipfile.ZipInfo] = {}
            for info in file_entries:
                if past_deadline() or stop_early:
                    break
                f = pool.submit(
                    extract_member, meta.path, info, dest_dir,
                    strip_top, top_folder, meta, pbar_zip, password
                )
                futures[f] = info

            for future in as_completed(futures):
                try:
                    r = future.result(timeout=300)
                    if r["status"] == "ok":
                        zip_files_done += 1
                    elif r["status"].startswith("skip"):
                        zip_files_skipped += 1
                    elif r["status"] in ("error", "blocked"):
                        zip_files_failed += 1
                except Exception as exc:
                    zip_files_failed += 1
                    info = futures[future]
                    logger.error("Worker exception for %s: %s", info.filename, exc)
                    with stats.lock:
                        stats.failures.append({
                            "zip": meta.path,
                            "name": info.filename,
                            "error": str(exc)
                        })

                # Heartbeat
                now = time.monotonic()
                if now - last_heartbeat > 30:
                    elapsed = now - RUN_START
                    speed = stats.bytes_done / elapsed if elapsed > 0 else 0
                    eta_s = (pbar_total.total - stats.bytes_done) / speed if speed > 0 else 0
                    print(f"  💓 {stats.files_done + stats.files_skipped} files | "
                          f"{human(stats.bytes_done)}/{human(pbar_total.total)} | "
                          f"{human(speed)}/s | ETA {eta_s/60:.0f} min | "
                          f"disk {human(free_local_bytes())}")
                    last_heartbeat = now

                if now - last_state_save > 30:
                    try:
                        os.makedirs(STATE_DIR, exist_ok=True)
                        state_data["last_update"] = datetime.datetime.now().isoformat()
                        atomic_json_write(STATE_FILE, state_data)
                    except Exception:
                        pass
                    last_state_save = now

                if past_deadline():
                    stop_early = True

    pbar_zip.close()

    result["files_done"] = zip_files_done
    result["files_skipped"] = zip_files_skipped
    result["files_failed"] = zip_files_failed
    result["time_s"] = time.monotonic() - t0

    if stop_early:
        result["status"] = "incomplete"
        result["error"] = "stopped early (time/disk)"
    elif zip_files_failed > 0:
        result["status"] = "partial"
        result["error"] = f"{zip_files_failed} entries failed"
    else:
        result["status"] = "ok"
        # Mark as done in state
        zip_key = os.path.realpath(meta.path)
        state_data.setdefault("done_zips", {})[zip_key] = {
            "time": datetime.datetime.now().isoformat(),
            "entries": meta.entry_count,
        }
        with stats.lock:
            stats.zips_done += 1

    pbar_total.update(max(0, meta.total_uncompressed - pbar_zip.n))

    return result


# ═════════════════════  MAIN EXTRACTION LOOP  ═════════════════════

_all_results = []
_stop_reason = _GDUZ_STOP_REASON if _GDUZ_STOP else ""
_skip_extraction = _GDUZ_STOP or (bytes_to_extract == 0 and not zip_metas)

if _skip_extraction:
    if _GDUZ_STOP:
        print(f"⚠️ Skipping extraction: {_GDUZ_STOP_REASON}")
    else:
        print("ℹ️ Nothing to extract.")

# Define these so the finally block and cell 6 don't crash
todo_metas = [m for m in zip_metas if m.state != "done" and not m.error] if not _skip_extraction else []
done_metas = [m for m in zip_metas if m.state == "done"] if not _skip_extraction else []
error_metas = [m for m in zip_metas if m.error and m.state != "done"] if not _skip_extraction else []

try:
    # Pre-done results
    for m in done_metas:
        _all_results.append({
            "zip": m.path, "name": os.path.basename(m.path),
            "entries": m.entry_count, "size": m.size,
            "uncompressed": m.total_uncompressed,
            "status": "already_done", "error": "",
            "time_s": 0, "files_done": 0,
            "files_skipped": m.entry_count, "files_failed": 0,
        })

    for m in error_metas:
        _all_results.append({
            "zip": m.path, "name": os.path.basename(m.path),
            "entries": m.entry_count, "size": m.size,
            "uncompressed": m.total_uncompressed,
            "status": "error", "error": m.error,
            "time_s": 0, "files_done": 0,
            "files_skipped": 0, "files_failed": m.entry_count,
        })
        with stats.lock:
            stats.zips_failed += 1

    if not todo_metas:
        print("✅ All zips already processed.")
    else:
        total_bytes_todo = sum(m.total_uncompressed for m in todo_metas)
        pbar_total = tqdm(total=total_bytes_todo, unit="B", unit_scale=True,
                          desc="Total progress", position=0)

        for meta in todo_metas:
            if past_deadline():
                _stop_reason = "time budget exceeded"
                logger.warning("⏰ Time budget exceeded. Run the notebook again to continue.")
                # Mark remaining as incomplete
                for rm in todo_metas[todo_metas.index(meta):]:
                    _all_results.append({
                        "zip": rm.path, "name": os.path.basename(rm.path),
                        "entries": rm.entry_count, "size": rm.size,
                        "uncompressed": rm.total_uncompressed,
                        "status": "not_started", "error": "time budget exceeded",
                        "time_s": 0, "files_done": 0,
                        "files_skipped": 0, "files_failed": 0,
                    })
                break

            if not check_local_disk():
                _stop_reason = "local disk full"
                logger.error("💾 Local disk critically low. Run again after freeing space.")
                break

            result = process_zip(meta, pbar_total)
            _all_results.append(result)

            # Handle nested zips
            if EXTRACT_NESTED_ZIPS and result["status"] in ("ok", "ok_7z", "ok_pyzipper"):
                nested = discover_zips([meta.dest_dir], True, ZIP_GLOB)
                for nz in nested:
                    if nz == meta.path:
                        continue
                    stats.nested_zips_found.append(nz)
                    logger.info("🔄 Found nested zip: %s", os.path.basename(nz))
                    # Only process if under depth limit
                    # (simplified: count nested by path depth relative to original dest)
                    nm = analyse_zip(nz)
                    nm.dest_dir = compute_dest(nm)
                    nm.state = check_zip_state(nm, state_data)
                    if nm.state != "done":
                        nr = process_zip(nm, pbar_total)
                        _all_results.append(nr)

            # Save state periodically
            try:
                os.makedirs(STATE_DIR, exist_ok=True)
                state_data["last_update"] = datetime.datetime.now().isoformat()
                atomic_json_write(STATE_FILE, state_data)
            except Exception:
                pass

        pbar_total.close()

except KeyboardInterrupt:
    _stop_reason = "keyboard interrupt"
    logger.warning("⚠️ KeyboardInterrupt — cleaning up …")
except Exception as exc:
    _stop_reason = f"error: {exc}"
    logger.error("❌ Unexpected error: %s", exc)
    traceback.print_exc()
finally:
    # Clean up .part files created in this run
    with stats.lock:
        for part in list(stats.part_files_created):
            try:
                if os.path.exists(part):
                    os.unlink(part)
                    logger.info("Cleaned up: %s", part)
            except OSError:
                pass
    os_sync()

    # Final state save
    try:
        os.makedirs(STATE_DIR, exist_ok=True)
        state_data["last_update"] = datetime.datetime.now().isoformat()
        atomic_json_write(STATE_FILE, state_data)
    except Exception:
        pass

    stats.zip_results = OrderedDict((r["name"], r) for r in _all_results)

    elapsed = time.monotonic() - RUN_START
    print(f"\n⏱️  Extraction phase done in {elapsed/60:.1f} min.")
    if _stop_reason:
        print(f"⚠️  Stopped early: {_stop_reason}")
    print(f"   Files extracted: {stats.files_done}  |  Skipped: {stats.files_skipped}  |  Failed: {stats.files_failed}")
    print(f"   Data processed: {human(stats.bytes_done)}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 6 — Verify + Cleanup + Report
# ══════════════════════════════════════════════════════════════════════════

print("=" * 90)
print("📊 VERIFICATION & CLEANUP")
print("=" * 90)

# ── Verify all extracted files ──
verify_ok = True
leftover_count = 0
leftovers_cleaned = 0

for meta in zip_metas:
    zip_name = os.path.basename(meta.path)
    dest_dir = meta.dest_dir
    zip_key = os.path.realpath(meta.path)

    if not os.path.isdir(dest_dir):
        # Not extracted yet
        if zip_key not in state_data.get("done_zips", {}):
            continue

    # Check for .part files in dest
    if os.path.isdir(dest_dir):
        for root, dirs, files in os.walk(dest_dir):
            dirs[:] = [d for d in dirs if not d.startswith(".")]
            for f in files:
                if f.endswith(".part"):
                    fp = os.path.join(root, f)
                    logger.warning("Removing leftover .part file: %s", fp)
                    try:
                        os.unlink(fp)
                        leftovers_cleaned += 1
                    except OSError as exc:
                        logger.error("Could not remove %s: %s", fp, exc)
                        leftover_count += 1

    # Skip verification for fallback-extracted zips or errored zips
    result_entry = stats.zip_results.get(zip_name, {})
    if result_entry.get("status", "").startswith("error"):
        continue
    if result_entry.get("status", "") in ("not_started",):
        continue
    if result_entry.get("status", "") in ("ok_7z",):
        logger.info("Skipping per-entry verification for 7z-extracted %s (coarser)", zip_name)
        continue

    # Per-entry verification
    if not meta.entries or meta.error:
        continue

    strip_top = (STRIP_SINGLE_TOP_FOLDER and meta.has_single_top_folder
                 and DEST_MODE != "same_folder")
    top_folder = ""
    if strip_top and meta.entries:
        top_folder = safe_name(decode_entry_name(meta.entries[0], ZIP_NAME_ENCODING)).split("/")[0]

    missing = 0
    size_mismatch = 0

    for info in meta.entries:
        if info.is_dir():
            name = decode_entry_name(info, ZIP_NAME_ENCODING)
            sname = safe_name(name)
            if not sname:
                continue
            out_name = sname
            if strip_top and top_folder:
                parts = out_name.split("/", 1)
                if len(parts) > 1:
                    out_name = parts[1]
                else:
                    continue
            dp = os.path.join(dest_dir, out_name)
            if not os.path.isdir(dp):
                # Try to create it
                try:
                    os.makedirs(dp, exist_ok=True)
                except Exception:
                    missing += 1
            continue

        name = decode_entry_name(info, ZIP_NAME_ENCODING)
        sname = safe_name(name)
        if not sname:
            continue
        out_name = sname
        if strip_top and top_folder:
            parts = out_name.split("/", 1)
            if len(parts) > 1:
                out_name = parts[1]
            else:
                out_name = parts[0]

        out_path = os.path.join(dest_dir, out_name)
        if not os.path.exists(out_path):
            missing += 1
            if result_entry.get("status") in ("ok", "already_done", "ok_pyzipper"):
                logger.warning("Missing after extraction: %s", out_path)
                verify_ok = False
        else:
            try:
                if os.path.getsize(out_path) != info.file_size:
                    size_mismatch += 1
                    verify_ok = False
                    logger.warning("Size mismatch: %s (got %s, expected %s)",
                                   out_path, os.path.getsize(out_path), info.file_size)
            except OSError:
                pass

    if missing > 0 and result_entry.get("status") in ("ok", "already_done", "ok_pyzipper"):
        logger.warning("%s: %d entries missing, %d size mismatches", zip_name, missing, size_mismatch)

# ── DELETE_ZIP_AFTER_SUCCESS ──
deleted_zips = []
if DELETE_ZIP_AFTER_SUCCESS:
    print("\n🗑️  Checking which zips can be deleted …")
    for meta in zip_metas:
        zip_key = os.path.realpath(meta.path)
        result_entry = stats.zip_results.get(os.path.basename(meta.path), {})
        status = result_entry.get("status", "")

        if status not in ("ok", "already_done", "ok_pyzipper"):
            continue

        # Re-verify this specific zip
        all_ok = True
        if meta.entries and not meta.error:
            strip_top = (STRIP_SINGLE_TOP_FOLDER and meta.has_single_top_folder
                         and DEST_MODE != "same_folder")
            top_folder = ""
            if strip_top and meta.entries:
                top_folder = safe_name(decode_entry_name(meta.entries[0], ZIP_NAME_ENCODING)).split("/")[0]

            for info in meta.entries:
                if info.is_dir():
                    continue
                name = decode_entry_name(info, ZIP_NAME_ENCODING)
                sname = safe_name(name)
                if not sname:
                    continue
                out_name = sname
                if strip_top and top_folder:
                    parts = out_name.split("/", 1)
                    if len(parts) > 1:
                        out_name = parts[1]
                    else:
                        out_name = parts[0]
                out_path = os.path.join(meta.dest_dir, out_name)
                if not os.path.exists(out_path):
                    all_ok = False
                    break
                try:
                    if os.path.getsize(out_path) != info.file_size:
                        all_ok = False
                        break
                except OSError:
                    all_ok = False
                    break

        if all_ok:
            try:
                os.unlink(meta.path)
                deleted_zips.append(meta.path)
                logger.info("🗑️  Deleted: %s", meta.path)
            except OSError as exc:
                logger.error("Could not delete %s: %s", meta.path, exc)

# ── Clean up state file if all done ──
all_complete = all(
    stats.zip_results.get(os.path.basename(m.path), {}).get("status", "")
    in ("ok", "already_done", "ok_pyzipper", "ok_7z")
    for m in zip_metas
)

if all_complete:
    # Remove state file and directory
    try:
        if os.path.exists(STATE_FILE):
            os.unlink(STATE_FILE)
        if os.path.isdir(STATE_DIR):
            try:
                os.rmdir(STATE_DIR)
            except OSError:
                pass  # Not empty = other state files
        logger.info("State file cleaned up.")
    except OSError:
        pass

# ── Clean up local temp files ──
local_content = "/content" if os.path.isdir("/content") else tempfile.gettempdir()
for root, dirs, files in os.walk(local_content):
    for f in files:
        if f.endswith(".part"):
            fp = os.path.join(root, f)
            try:
                os.unlink(fp)
                leftovers_cleaned += 1
            except OSError:
                leftover_count += 1

# ══════════════════════════════════════════════════════════════════════════
# FINAL REPORT
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 90)
print("📋 FINAL REPORT")
print("=" * 90)

total_elapsed = time.monotonic() - RUN_START

print(f"\n{'ZIP FILE':<35} {'ENTRIES':>8} {'SIZE':>10} {'UNCOMPRESSED':>12} {'TIME':>8} {'STATUS':>15}")
print("-" * 90)
for r in _all_results:
    name = r["name"]
    if len(name) > 33:
        name = name[:30] + "…"
    t = f"{r['time_s']:.0f}s" if r['time_s'] > 0 else "—"
    status_icon = {
        "ok": "✅", "already_done": "✅", "ok_7z": "✅",
        "ok_pyzipper": "✅", "partial": "⚠️", "incomplete": "🔄",
        "not_started": "⏳", "error": "❌", "error_7z": "❌",
        "error_pyzipper": "❌",
    }.get(r["status"], "❓")
    print(f"{name:<35} {r['entries']:>8} {human(r['size']):>10} {human(r['uncompressed']):>12} {t:>8} {status_icon} {r['status']}")
    if r["error"]:
        print(f"{'':>35} ↳ {r['error']}")

print("-" * 90)
print(f"Total elapsed: {total_elapsed/60:.1f} min")
print(f"Files extracted: {stats.files_done}  |  Skipped (already done): {stats.files_skipped}  |  Failed: {stats.files_failed}")
print(f"Dirs created: {stats.dirs_created}  |  Entries renamed: {stats.files_renamed}")
print(f"Data processed: {human(stats.bytes_done)}")
print(f"Zips completed: {stats.zips_done}  |  Zips failed: {stats.zips_failed}")

if leftovers_cleaned > 0:
    print(f"🧹 Cleaned up {leftovers_cleaned} leftover .part files")
if leftover_count > 0:
    print(f"⚠️  {leftover_count} leftover files could not be removed")
else:
    print(f"✅ No leftover .part or temp files remain (count: {leftover_count})")

if deleted_zips:
    print(f"\n🗑️  Deleted {len(deleted_zips)} source zip(s):")
    for dz in deleted_zips:
        print(f"   {dz}")
    print("   ⚠️  Remember: deleted files still use storage until you empty the Trash!")

if stats.failures:
    print(f"\n❌ FAILURES ({len(stats.failures)}):")
    for fail in stats.failures[:50]:
        print(f"   {os.path.basename(fail['zip'])}: {fail['name']} — {fail['error']}")
    if len(stats.failures) > 50:
        print(f"   … and {len(stats.failures) - 50} more")

if stats.renamed_entries:
    n = len(stats.renamed_entries)
    print(f"\n📝 Renamed entries: {n}")
    for re_entry in stats.renamed_entries[:20]:
        print(f"   {re_entry['original']} → {re_entry['safe']}")
    if n > 20:
        print(f"   … and {n - 20} more")

if stats.skipped_entries:
    print(f"\n⏭️  Skipped entries: {len(stats.skipped_entries)}")
    for sk in stats.skipped_entries[:10]:
        print(f"   {sk['name']} — {sk['reason']}")

if stats.nested_zips_found:
    print(f"\n🔄 Nested zips processed: {len(stats.nested_zips_found)}")

# Final verdict
print("\n" + "=" * 90)
if all_complete and leftover_count == 0 and stats.files_failed == 0:
    print("🎉 ALL DONE — nothing left behind.")
elif _stop_reason:
    print(f"🔄 INCOMPLETE — run again to continue. Reason: {_stop_reason}")
else:
    if stats.files_failed > 0:
        print(f"⚠️  COMPLETED WITH ERRORS — {stats.files_failed} entries failed. Check the failure list above.")
    else:
        print("🔄 INCOMPLETE — run again to continue.")
print("=" * 90)

# ── Save CSV report ──
if SAVE_REPORT:
    report_path = os.path.join(DRIVE_MOUNT, "MyDrive", "unzip_report.csv")
    try:
        with open(report_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=[
                "zip", "name", "entries", "size", "uncompressed",
                "status", "error", "time_s", "files_done", "files_skipped", "files_failed"
            ])
            writer.writeheader()
            for r in _all_results:
                writer.writerow(r)
        print(f"\n📄 Report saved: {report_path}")
    except Exception as exc:
        logger.error("Could not save report: %s", exc)

# ── Unmount ──
if UNMOUNT_AT_END and IN_COLAB and all_complete:
    try:
        from google.colab import drive
        drive.flush_and_unmount()
        print("🔌 Drive unmounted.")
    except Exception as exc:
        logger.warning("Unmount failed: %s", exc)
elif UNMOUNT_AT_END and IN_COLAB and not all_complete:
    print("ℹ️  Not unmounting — extraction is incomplete. Run again to continue.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 7 — Standalone cleanup: remove stale .part files and state
# ══════════════════════════════════════════════════════════════════════════
# Safe to run at any time. Only touches destination folders and the state file.

print("🧹 Standalone cleanup — scanning for leftovers …\n")

_cleanup_count = 0

# Clean .part files in destination folders
for meta in zip_metas:
    dest = meta.dest_dir
    if not os.path.isdir(dest):
        continue
    for root, dirs, files in os.walk(dest):
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        for f in files:
            if f.endswith(".part"):
                fp = os.path.join(root, f)
                try:
                    os.unlink(fp)
                    print(f"  Removed: {fp}")
                    _cleanup_count += 1
                except OSError as exc:
                    print(f"  Could not remove: {fp} ({exc})")

# Clean state file
if os.path.exists(STATE_FILE):
    try:
        os.unlink(STATE_FILE)
        print(f"  Removed state file: {STATE_FILE}")
        _cleanup_count += 1
    except OSError as exc:
        print(f"  Could not remove state file: {exc}")

if os.path.isdir(STATE_DIR):
    try:
        os.rmdir(STATE_DIR)
        print(f"  Removed state dir: {STATE_DIR}")
    except OSError:
        pass  # Not empty

if _cleanup_count == 0:
    print("✅ No leftovers found — everything is clean.")
else:
    print(f"\n🧹 Cleaned up {_cleanup_count} item(s).")
